# $1\mu1p$ selection test

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import uproot

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.file_locations import intermediate_files_location

root_path = Path(f"{intermediate_files_location}/minimal_withspline_df.root")
tree_name = "tree"

assert root_path.exists(), f"Missing ROOT file: {root_path}"

In [ ]:
with uproot.open(root_path) as f:
    print(f.keys())
    tree = f[tree_name]
    print(f"{tree_name}: {tree.num_entries:,} entries")
    afro_branches = [name for name in tree.keys() if name.startswith("afro_1mu1p_")]
    afro_branches

In [ ]:
branches = ["afro_1mu1p_sel", "afro_1mu1p_Pn"]

arrays = uproot.open(root_path)[tree_name].arrays(branches, library="np")
sel = arrays["afro_1mu1p_sel"].astype(bool)
pn = arrays["afro_1mu1p_Pn"]

# The selection code fills invalid/uncomputed STV values with -9999.0.
valid_pn = np.isfinite(pn) & (pn > -9990)
selected_pn = pn[sel & valid_pn]

print(f"Total events: {len(pn):,}")
print(f"AFro 1mu1p selected events: {sel.sum():,}")
print(f"Selected events with valid Pn: {len(selected_pn):,}")

if len(selected_pn):
    print(
        "Pn summary [MeV/c]: "
        f"min={selected_pn.min():.2f}, "
        f"median={np.median(selected_pn):.2f}, "
        f"max={selected_pn.max():.2f}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(selected_pn, bins=20, range=(0, 1.0), histtype="step", linewidth=2)
ax.set_xlabel(r"$p_n$")
ax.set_ylabel("Events")
ax.set_title("$1\\mu1p$ selected")
ax.grid(alpha=0.25)

fig.tight_layout()